# Ni–Co Metal Oxide / Graphene Supercapacitor Electrode
# Physics-Informed AI Modelling & Multi-Objective Optimization

This notebook implements **Step 7 (Physics-Informed AI Modelling and Optimization)** and
**Step 8 (Experimental Validation)** of the workflow:

1. Nickel–Cobalt oxide precursor synthesis (Steps 1–3)
2. Graphene nanocomposite formation (Step 4)
3. Electrode fabrication (Step 5)
4. Material/electrochemical characterization — XRD, SEM/TEM, EDS, Raman, BET, CV, GCD, EIS (Step 6)
5. **Physics-informed ML/DL modelling of specific capacitance, energy density, power density,
   rate capability and cycling stability, followed by multi-objective optimization (Step 7)**
6. **Experimental validation of the AI-predicted optimum against real measurements (Step 8)**

> **Important — synthetic data notice.** Steps 1–6 are physical laboratory procedures and cannot
> be "run" in a notebook. This notebook therefore ships with a **physically-motivated synthetic
> dataset generator** (Section 1) that stands in for your real experimental data so that every
> cell below executes end-to-end out of the box. **Replace the synthetic dataset with your real
> Step 1–6 measurements** by loading a CSV with the same column schema — see Section 1.2.
> Everything downstream (feature engineering, ML/PINN models, optimization, validation) is a
> genuine, runnable implementation, not a placeholder.

**Environment:** pure NumPy / pandas / scikit-learn / SciPy / Matplotlib / Seaborn — no GPU or
internet access required.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.optimize import differential_evolution

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## Section 1 — Experimental Data (Steps 1–6 outputs)

### 1.1 Data schema

| column | meaning | typical range |
|---|---|---|
| `ni_co_ratio` | Ni / (Ni+Co) molar fraction | 0.3 – 0.9 |
| `precursor_ph` | NaOH-controlled precipitation pH | 9 – 12 |
| `hydrothermal_temp_C` | sealed-vessel reaction temperature | 120 – 200 °C |
| `hydrothermal_time_h` | sealed-vessel reaction time | 6 – 24 h |
| `calcination_temp_C` | calcination temperature | 300 – 600 °C |
| `calcination_time_h` | calcination holding time | 1 – 4 h |
| `graphene_wt_pct` | graphene loading in nanocomposite | 0 – 15 wt% |
| `bet_surface_area_m2g` | BET surface area (characterization) | 40 – 220 m²/g |
| `crystallite_size_nm` | Scherrer crystallite size (XRD) | 5 – 45 nm |
| `conductivity_S_cm` | composite electrical conductivity | — |
| `specific_capacitance_F_g` | **target** — GCD specific capacitance | 100 – 1400 F/g |
| `energy_density_Wh_kg` | **target** — derived from CV/GCD | — |
| `power_density_W_kg` | **target** | — |
| `capacity_retention_pct` | **target** — after N cycles (EIS/GCD cycling) | 60 – 100 % |
| `voltage_window_V` | operating voltage window used in E,P calc | 0.4 – 1.6 V |

### 1.2 Loading real data
Once you have real Step 1–6 results, replace the synthetic generator call below with:
```python
df = pd.read_csv("your_experimental_data.csv")
```
using the exact column names in the table above.


In [ ]:
def generate_synthetic_dataset(n_samples: int = 320, seed: int = RANDOM_STATE) -> pd.DataFrame:
    """
    Physically-motivated synthetic stand-in for Step 1-6 experimental output.

    The functional forms below are NOT literature-fitted constants; they encode
    qualitative, well-known trends reported for Ni-Co oxide/graphene supercapacitor
    electrodes (volcano-shaped capacitance vs. calcination temperature, capacitance
    gains that saturate with graphene loading, capacitance-energy consistency via
    E = 0.5*C*V^2/3.6, etc.) purely so the rest of the notebook has realistic,
    internally-consistent data to model. Replace with real measurements for
    production use.
    """
    g = np.random.default_rng(seed)

    ni_co_ratio          = g.uniform(0.3, 0.9, n_samples)
    precursor_ph         = g.uniform(9.0, 12.0, n_samples)
    hydrothermal_temp_C  = g.uniform(120, 200, n_samples)
    hydrothermal_time_h  = g.uniform(6, 24, n_samples)
    calcination_temp_C   = g.uniform(300, 600, n_samples)
    calcination_time_h   = g.uniform(1, 4, n_samples)
    graphene_wt_pct      = g.uniform(0, 15, n_samples)
    voltage_window_V     = g.uniform(0.4, 1.6, n_samples)

    # --- characterization proxies (physics-informed intermediate features) ---
    bet_surface_area_m2g = (
        60
        + 1.1 * graphene_wt_pct * (1 - graphene_wt_pct / 40)     # saturating gain
        + 0.15 * hydrothermal_time_h
        - 0.05 * (calcination_temp_C - 450)                       # sintering closes pores
        + g.normal(0, 6, n_samples)
    ).clip(20, 260)

    crystallite_size_nm = (
        6
        + 0.045 * (calcination_temp_C - 300)                       # Ostwald ripening
        + 1.2 * calcination_time_h
        - 0.02 * hydrothermal_time_h
        + g.normal(0, 1.2, n_samples)
    ).clip(3, 60)

    conductivity_S_cm = (
        0.5 + 4.5 * (1 - np.exp(-graphene_wt_pct / 4.0)) + g.normal(0, 0.15, n_samples)
    ).clip(0.05, None)

    # --- electrochemical targets ---
    # volcano response of the Ni:Co ratio around ~0.65 (redox-active balance)
    ratio_term = 1 - 3.0 * (ni_co_ratio - 0.65) ** 2
    # volcano response of crystallite size (too small = unstable, too large = low active area)
    size_term = np.exp(-((crystallite_size_nm - 14) ** 2) / (2 * 9 ** 2))

    specific_capacitance_F_g = (
        300
        + 5.5 * bet_surface_area_m2g * size_term
        + 260 * ratio_term
        + 60 * np.log1p(conductivity_S_cm)
        + g.normal(0, 35, n_samples)
    ).clip(50, None)

    # theoretical Ni-Co redox ceiling used later as a physics bound
    theoretical_cap_max = 1500.0

    energy_density_Wh_kg = (
        0.5 * specific_capacitance_F_g * voltage_window_V ** 2 / 3.6
        + g.normal(0, 1.5, n_samples)
    ).clip(0, None)

    power_density_W_kg = (
        (voltage_window_V ** 2) / (4 * (1.2 - 0.15 * np.log1p(conductivity_S_cm))) * 1000
        + g.normal(0, 25, n_samples)
    ).clip(0, None)

    capacity_retention_pct = (
        100
        - 0.18 * crystallite_size_nm
        - 0.9 * (graphene_wt_pct < 2) * (2 - graphene_wt_pct)   # too little graphene -> poor buffering
        + 0.05 * graphene_wt_pct
        + g.normal(0, 2.0, n_samples)
    ).clip(40, 100)

    df = pd.DataFrame({
        "ni_co_ratio": ni_co_ratio,
        "precursor_ph": precursor_ph,
        "hydrothermal_temp_C": hydrothermal_temp_C,
        "hydrothermal_time_h": hydrothermal_time_h,
        "calcination_temp_C": calcination_temp_C,
        "calcination_time_h": calcination_time_h,
        "graphene_wt_pct": graphene_wt_pct,
        "voltage_window_V": voltage_window_V,
        "bet_surface_area_m2g": bet_surface_area_m2g,
        "crystallite_size_nm": crystallite_size_nm,
        "conductivity_S_cm": conductivity_S_cm,
        "specific_capacitance_F_g": specific_capacitance_F_g,
        "energy_density_Wh_kg": energy_density_Wh_kg,
        "power_density_W_kg": power_density_W_kg,
        "capacity_retention_pct": capacity_retention_pct,
    })
    df.attrs["theoretical_cap_max"] = theoretical_cap_max
    return df

df = generate_synthetic_dataset()
THEORETICAL_CAP_MAX = df.attrs["theoretical_cap_max"]
df.head()


In [ ]:
df.describe().T.round(2)

## Section 2 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, ax=axes[0])
axes[0].set_title("Correlation matrix")

sns.scatterplot(
    data=df, x="bet_surface_area_m2g", y="specific_capacitance_F_g",
    hue="graphene_wt_pct", palette="viridis", ax=axes[1]
)
axes[1].set_title("Specific capacitance vs BET surface area")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_overview.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.scatterplot(data=df, x="calcination_temp_C", y="specific_capacitance_F_g",
                 hue="crystallite_size_nm", palette="mako", ax=axes[0])
axes[0].set_title("Capacitance vs calcination temperature")

sns.scatterplot(data=df, x="ni_co_ratio", y="specific_capacitance_F_g",
                 ax=axes[1], color="teal")
axes[1].set_title("Capacitance vs Ni/Co ratio (volcano trend)")

sns.scatterplot(data=df, x="graphene_wt_pct", y="capacity_retention_pct",
                 ax=axes[2], color="darkorange")
axes[2].set_title("Cycling retention vs graphene wt%")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_trends.png", bbox_inches="tight")
plt.show()


## Section 3 — Physics-informed feature engineering

We add features grounded in electrochemistry rather than purely statistical ones:

* **Areal utilization** `C / BET` — how efficiently the accessible surface area is being converted
  into capacitance (charge-transfer efficiency proxy).
* **Ni/Co deviation** from the redox-optimal ratio (~0.65 for spinel NiCo₂O₄-type phases).
* **Crystallinity index** — normalized crystallite size, since electrochemical activity is
  known to peak at an intermediate grain size (enough crystallinity for conductivity, enough
  defects/edges for redox sites).
* **Graphene saturation term** — `1 - exp(-graphene_wt%/4)`, capturing the diminishing-returns
  conductivity boost widely reported for graphene loading.


In [ ]:
df_fe = df.copy()

df_fe["areal_utilization_F_m2"] = df_fe["specific_capacitance_F_g"] / df_fe["bet_surface_area_m2g"]
df_fe["ni_co_deviation"] = np.abs(df_fe["ni_co_ratio"] - 0.65)
df_fe["crystallinity_index"] = np.exp(-((df_fe["crystallite_size_nm"] - 14) ** 2) / (2 * 9 ** 2))
df_fe["graphene_saturation"] = 1 - np.exp(-df_fe["graphene_wt_pct"] / 4.0)
df_fe["theoretical_cap_max"] = THEORETICAL_CAP_MAX

PROCESS_FEATURES = [
    "ni_co_ratio", "precursor_ph", "hydrothermal_temp_C", "hydrothermal_time_h",
    "calcination_temp_C", "calcination_time_h", "graphene_wt_pct", "voltage_window_V",
]
CHAR_FEATURES = ["bet_surface_area_m2g", "crystallite_size_nm", "conductivity_S_cm"]
ENGINEERED_FEATURES = ["ni_co_deviation", "crystallinity_index", "graphene_saturation"]

FEATURE_COLUMNS = PROCESS_FEATURES + CHAR_FEATURES + ENGINEERED_FEATURES
TARGET_COLUMNS = [
    "specific_capacitance_F_g", "energy_density_Wh_kg",
    "power_density_W_kg", "capacity_retention_pct",
]

df_fe[FEATURE_COLUMNS + TARGET_COLUMNS].head()


## Section 4 — Train / test split & scaling

In [ ]:
X = df_fe[FEATURE_COLUMNS].values
Y = df_fe[TARGET_COLUMNS].values

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=RANDOM_STATE
)

x_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(Y_train)

X_train_s = x_scaler.transform(X_train)
X_test_s = x_scaler.transform(X_test)
Y_train_s = y_scaler.transform(Y_train)
Y_test_s = y_scaler.transform(Y_test)

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples | Features: {X.shape[1]}")


## Section 5 — Baseline ML models (Random Forest & Gradient Boosting)

Two tree-ensemble multi-output regressors are trained as strong, easy-to-trust baselines
before introducing the physics-informed network.

In [ ]:
def evaluate(name, y_true, y_pred, target_names=TARGET_COLUMNS):
    rows = []
    for i, t in enumerate(target_names):
        rows.append({
            "model": name, "target": t,
            "R2": r2_score(y_true[:, i], y_pred[:, i]),
            "RMSE": np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])),
            "MAE": mean_absolute_error(y_true[:, i], y_pred[:, i]),
        })
    return pd.DataFrame(rows)

rf = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=400, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1)
)
rf.fit(X_train_s, Y_train_s)
rf_pred = y_scaler.inverse_transform(rf.predict(X_test_s))

gbr = MultiOutputRegressor(
    GradientBoostingRegressor(n_estimators=400, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE)
)
gbr.fit(X_train_s, Y_train_s)
gbr_pred = y_scaler.inverse_transform(gbr.predict(X_test_s))

results = pd.concat([
    evaluate("RandomForest", Y_test, rf_pred),
    evaluate("GradientBoosting", Y_test, gbr_pred),
], ignore_index=True)
results.round(3)


In [ ]:
importances = pd.DataFrame(
    {est.feature_importances_.round(3) for est in []}
) if False else None  # placeholder cleared below

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, (t, ax) in enumerate(zip(TARGET_COLUMNS, axes)):
    imp = rf.estimators_[i].feature_importances_
    order = np.argsort(imp)[::-1][:6]
    ax.barh(np.array(FEATURE_COLUMNS)[order][::-1], imp[order][::-1], color="steelblue")
    ax.set_title(t, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rf_feature_importance.png", bbox_inches="tight")
plt.show()


## Section 6 — Physics-Informed Neural Network (PINN)

A compact single-hidden-layer network (`tanh` activation, manually implemented in NumPy —
no `torch`/`tensorflow` dependency) jointly predicts **specific capacitance** and
**energy density**, trained with a loss that combines:

1. **Data loss** — standard MSE against measured capacitance/energy.
2. **Physics-consistency loss** — enforces the thermodynamic relationship
   $E = \dfrac{0.5 \, C \, V^2}{3.6}$ between the two predicted outputs (Wh/kg from F/g and V).
3. **Bound loss** — a hinge penalty keeping predicted capacitance within
   $[0,\ C_{\text{theoretical,max}}]$, the Ni–Co redox ceiling.

Training uses full-batch gradient descent with the **Adam** optimizer and hand-derived
analytic gradients (verified below with a numerical-gradient check).

In [ ]:
class PINN:
    """Single-hidden-layer physics-informed network, predicting [capacitance, energy_density]."""

    def __init__(self, n_in, n_hidden=16, seed=RANDOM_STATE):
        g = np.random.default_rng(seed)
        limit1 = np.sqrt(6 / (n_in + n_hidden))
        limit2 = np.sqrt(6 / (n_hidden + 2))
        self.W1 = g.uniform(-limit1, limit1, (n_in, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.W2 = g.uniform(-limit2, limit2, (n_hidden, 2))
        self.b2 = np.zeros(2)

    def forward(self, X):
        Z1 = X @ self.W1 + self.b1
        A1 = np.tanh(Z1)
        Z2 = A1 @ self.W2 + self.b2   # [:,0]=capacitance (scaled), [:,1]=energy (scaled)
        cache = (X, Z1, A1)
        return Z2, cache

    def params(self):
        return [self.W1, self.b1, self.W2, self.b2]

    def set_params(self, params):
        self.W1, self.b1, self.W2, self.b2 = params


def pinn_loss_and_grads(model, X, C_true, E_true, V_window, cap_max_scaled,
                         c_scale, e_scale, c_mean, e_mean,
                         lambda_phys=0.3, lambda_bound=0.05):
    """
    Computes total loss and analytic gradients w.r.t. all PINN parameters.
    C_true, E_true are STANDARDIZED targets; predictions are un-standardized internally
    for the physics term (which must hold in physical, not standardized, units).
    """
    N = X.shape[0]
    Z2, (Xc, Z1, A1) = model.forward(X)
    C_pred_s, E_pred_s = Z2[:, 0], Z2[:, 1]

    # un-standardize for the physical consistency check
    C_pred = C_pred_s * c_scale + c_mean
    E_pred = E_pred_s * e_scale + e_mean

    # --- data loss (standardized space) ---
    dC_data = (2.0 / N) * (C_pred_s - C_true)
    dE_data = (2.0 / N) * (E_pred_s - E_true)
    L_data = np.mean((C_pred_s - C_true) ** 2) + np.mean((E_pred_s - E_true) ** 2)

    # --- physics consistency: E = 0.5*C*V^2/3.6 (physical units) ---
    E_from_C = 0.5 * C_pred * V_window ** 2 / 3.6
    resid = E_pred - E_from_C
    L_phys = np.mean(resid ** 2)
    # d(resid)/dC_pred = -0.5*V^2/3.6 ; d(resid)/dE_pred = 1
    dresid_dCpred = -0.5 * V_window ** 2 / 3.6
    dL_phys_dCpred = (2.0 / N) * resid * dresid_dCpred
    dL_phys_dEpred = (2.0 / N) * resid * 1.0
    # chain through un-standardization: dCpred/dCpred_s = c_scale, dEpred/dEpred_s = e_scale
    dphys_dCs = dL_phys_dCpred * c_scale
    dphys_dEs = dL_phys_dEpred * e_scale

    # --- bound penalty on capacitance (physical units): 0 <= C_pred <= cap_max ---
    cap_max = cap_max_scaled  # already physical units (theoretical max, e.g. 1500 F/g)
    over = np.maximum(C_pred - cap_max, 0.0)
    under = np.maximum(-C_pred, 0.0)
    L_bound = np.mean(over ** 2) + np.mean(under ** 2)
    dbound_dCpred = (2.0 / N) * over * 1.0 + (2.0 / N) * under * (-1.0)
    dbound_dCs = dbound_dCpred * c_scale

    L = L_data + lambda_phys * L_phys + lambda_bound * L_bound

    dZ2 = np.zeros_like(Z2)
    dZ2[:, 0] = dC_data + lambda_phys * dphys_dCs + lambda_bound * dbound_dCs
    dZ2[:, 1] = dE_data + lambda_phys * dphys_dEs

    # --- backprop through linear output layer ---
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0)
    dA1 = dZ2 @ model.W2.T
    dZ1 = dA1 * (1 - np.tanh(Z1) ** 2)
    dW1 = Xc.T @ dZ1
    db1 = dZ1.sum(axis=0)

    grads = [dW1, db1, dW2, db2]
    return L, grads, (L_data, L_phys, L_bound)


In [ ]:
# --- numerical-gradient sanity check (verifies the hand-derived backprop above) ---
def flatten(params):
    return np.concatenate([p.ravel() for p in params])

def unflatten(flat, shapes):
    out, idx = [], 0
    for shp in shapes:
        size = int(np.prod(shp))
        out.append(flat[idx:idx + size].reshape(shp))
        idx += size
    return out

_pinn_check = PINN(n_in=X_train_s.shape[1], n_hidden=8, seed=1)
_Xc = X_train_s[:20]
_Cc = Y_train_s[:20, 0]
_Ec = Y_train_s[:20, 1]
_Vc = X_train[:20, PROCESS_FEATURES.index("voltage_window_V")]
c_mean_, c_scale_ = y_scaler.mean_[0], y_scaler.scale_[0]
e_mean_, e_scale_ = y_scaler.mean_[1], y_scaler.scale_[1]

L0, grads0, _ = pinn_loss_and_grads(_pinn_check, _Xc, _Cc, _Ec, _Vc, THEORETICAL_CAP_MAX,
                                     c_scale_, e_scale_, c_mean_, e_mean_)

shapes = [p.shape for p in _pinn_check.params()]
flat_params = flatten(_pinn_check.params())
analytic_grad = flatten(grads0)

eps = 1e-5
numeric_grad = np.zeros_like(flat_params)
# check a random subset of parameters for speed
idxs = np.random.default_rng(0).choice(len(flat_params), size=25, replace=False)
for i in idxs:
    fp_plus = flat_params.copy(); fp_plus[i] += eps
    fp_minus = flat_params.copy(); fp_minus[i] -= eps
    m_plus = PINN(n_in=X_train_s.shape[1], n_hidden=8); m_plus.set_params(unflatten(fp_plus, shapes))
    m_minus = PINN(n_in=X_train_s.shape[1], n_hidden=8); m_minus.set_params(unflatten(fp_minus, shapes))
    Lp, _, _ = pinn_loss_and_grads(m_plus, _Xc, _Cc, _Ec, _Vc, THEORETICAL_CAP_MAX, c_scale_, e_scale_, c_mean_, e_mean_)
    Lm, _, _ = pinn_loss_and_grads(m_minus, _Xc, _Cc, _Ec, _Vc, THEORETICAL_CAP_MAX, c_scale_, e_scale_, c_mean_, e_mean_)
    numeric_grad[i] = (Lp - Lm) / (2 * eps)

rel_err = np.abs(numeric_grad[idxs] - analytic_grad[idxs]) / (np.abs(numeric_grad[idxs]) + 1e-8)
print(f"Max relative gradient error on {len(idxs)} sampled params: {rel_err.max():.2e}")
assert rel_err.max() < 1e-3, "Gradient check failed - backprop implementation has a bug."
print("Gradient check passed: analytic backprop matches numerical gradient.")


In [ ]:
def train_pinn(X, Y, V_window, cap_max, x_scaler, y_scaler, n_hidden=16,
               epochs=1500, lr=0.01, lambda_phys=0.3, lambda_bound=0.05, seed=RANDOM_STATE, verbose=True):
    model = PINN(n_in=X.shape[1], n_hidden=n_hidden, seed=seed)
    params = model.params()
    # Adam optimizer state
    m = [np.zeros_like(p) for p in params]
    v = [np.zeros_like(p) for p in params]
    beta1, beta2, eps_adam = 0.9, 0.999, 1e-8

    c_mean, c_scale = y_scaler.mean_[0], y_scaler.scale_[0]
    e_mean, e_scale = y_scaler.mean_[1], y_scaler.scale_[1]

    history = []
    for epoch in range(1, epochs + 1):
        L, grads, parts = pinn_loss_and_grads(
            model, X, Y[:, 0], Y[:, 1], V_window, cap_max,
            c_scale, e_scale, c_mean, e_mean, lambda_phys, lambda_bound
        )
        new_params = []
        for i, (p, g) in enumerate(zip(params, grads)):
            m[i] = beta1 * m[i] + (1 - beta1) * g
            v[i] = beta2 * v[i] + (1 - beta2) * (g ** 2)
            m_hat = m[i] / (1 - beta1 ** epoch)
            v_hat = v[i] / (1 - beta2 ** epoch)
            p = p - lr * m_hat / (np.sqrt(v_hat) + eps_adam)
            new_params.append(p)
        params = new_params
        model.set_params(params)
        history.append({"epoch": epoch, "loss": L, "data": parts[0], "phys": parts[1], "bound": parts[2]})
        if verbose and (epoch % 300 == 0 or epoch == 1):
            print(f"epoch {epoch:5d} | loss {L:.4f} | data {parts[0]:.4f} | phys {parts[1]:.4f} | bound {parts[2]:.4f}")

    return model, pd.DataFrame(history)

V_train = X_train[:, PROCESS_FEATURES.index("voltage_window_V")]
V_test = X_test[:, PROCESS_FEATURES.index("voltage_window_V")]

pinn_model, pinn_history = train_pinn(
    X_train_s, Y_train_s[:, :2], V_train, THEORETICAL_CAP_MAX, x_scaler, y_scaler
)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(pinn_history["epoch"], pinn_history["loss"], label="total")
ax.plot(pinn_history["epoch"], pinn_history["data"], label="data", alpha=0.7)
ax.plot(pinn_history["epoch"], pinn_history["phys"], label="physics-consistency", alpha=0.7)
ax.plot(pinn_history["epoch"], pinn_history["bound"], label="bound penalty", alpha=0.7)
ax.set_yscale("log")
ax.set_xlabel("epoch"); ax.set_ylabel("loss (log scale)")
ax.legend()
ax.set_title("PINN training curve")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pinn_training_curve.png", bbox_inches="tight")
plt.show()


In [ ]:
Z2_test, _ = pinn_model.forward(X_test_s)
pinn_C_pred = Z2_test[:, 0] * y_scaler.scale_[0] + y_scaler.mean_[0]
pinn_E_pred = Z2_test[:, 1] * y_scaler.scale_[1] + y_scaler.mean_[1]

pinn_pred_2 = np.column_stack([pinn_C_pred, pinn_E_pred])
pinn_eval = evaluate("PINN", Y_test[:, :2], pinn_pred_2, target_names=TARGET_COLUMNS[:2])
pd.concat([results[results.target.isin(TARGET_COLUMNS[:2])], pinn_eval], ignore_index=True).round(3)


## Section 7 — Predicted vs. actual (model comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for i, (t, ax) in enumerate(zip(TARGET_COLUMNS[:2], axes)):
    ax.scatter(Y_test[:, i], rf_pred[:, i], alpha=0.5, label="RandomForest", s=20)
    ax.scatter(Y_test[:, i], pinn_pred_2[:, i], alpha=0.5, label="PINN", s=20)
    lims = [min(Y_test[:, i].min(), rf_pred[:, i].min()), max(Y_test[:, i].max(), rf_pred[:, i].max())]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xlabel("actual"); ax.set_ylabel("predicted"); ax.set_title(t)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pred_vs_actual.png", bbox_inches="tight")
plt.show()


## Section 8 — Multi-Objective Optimization

We use the trained **Random Forest surrogate** (chosen here for its robustness on the
tabular process-parameter space; swap in the PINN or GBR by changing `SURROGATE` below) to
search the 8-dimensional process-parameter space for conditions that jointly maximize:

* specific capacitance
* energy density
* capacity retention (cycling stability)

This is a classic 3-objective trade-off (e.g. higher graphene loading helps retention but can
dilute volumetric capacitance). We trace the **Pareto front** with the weighted-sum method:
for many random objective weightings we solve a single-objective maximization with
`scipy.optimize.differential_evolution`, then keep only the **non-dominated** solutions.


In [ ]:
SURROGATE = rf  # trained MultiOutputRegressor; predicts all 4 targets from scaled features

PROCESS_BOUNDS = {
    "ni_co_ratio": (0.3, 0.9),
    "precursor_ph": (9.0, 12.0),
    "hydrothermal_temp_C": (120, 200),
    "hydrothermal_time_h": (6, 24),
    "calcination_temp_C": (300, 600),
    "calcination_time_h": (1, 4),
    "graphene_wt_pct": (0, 15),
    "voltage_window_V": (0.4, 1.6),
}
bounds = [PROCESS_BOUNDS[f] for f in PROCESS_FEATURES]

def build_full_feature_row(process_vals):
    """Given the 8 raw process parameters, reconstruct characterization + engineered
    features using the SAME physically-motivated relations as the data generator, so the
    surrogate (trained on the full 14-D feature space) receives consistent inputs."""
    (ni_co_ratio, precursor_ph, hydrothermal_temp_C, hydrothermal_time_h,
     calcination_temp_C, calcination_time_h, graphene_wt_pct, voltage_window_V) = process_vals

    bet_surface_area_m2g = np.clip(
        60 + 1.1 * graphene_wt_pct * (1 - graphene_wt_pct / 40)
        + 0.15 * hydrothermal_time_h - 0.05 * (calcination_temp_C - 450), 20, 260)
    crystallite_size_nm = np.clip(
        6 + 0.045 * (calcination_temp_C - 300) + 1.2 * calcination_time_h
        - 0.02 * hydrothermal_time_h, 3, 60)
    conductivity_S_cm = max(0.5 + 4.5 * (1 - np.exp(-graphene_wt_pct / 4.0)), 0.05)

    ni_co_deviation = abs(ni_co_ratio - 0.65)
    crystallinity_index = np.exp(-((crystallite_size_nm - 14) ** 2) / (2 * 9 ** 2))
    graphene_saturation = 1 - np.exp(-graphene_wt_pct / 4.0)

    row = [ni_co_ratio, precursor_ph, hydrothermal_temp_C, hydrothermal_time_h,
           calcination_temp_C, calcination_time_h, graphene_wt_pct, voltage_window_V,
           bet_surface_area_m2g, crystallite_size_nm, conductivity_S_cm,
           ni_co_deviation, crystallinity_index, graphene_saturation]
    return np.array(row)

def predict_objectives(process_vals):
    """Returns (capacitance, energy_density, capacity_retention) for a raw process vector."""
    row = build_full_feature_row(process_vals)
    row_s = x_scaler.transform(row.reshape(1, -1))
    pred_s = SURROGATE.predict(row_s)[0]
    pred = y_scaler.inverse_transform(pred_s.reshape(1, -1))[0]
    C, E, P, R = pred
    return C, E, R  # capacitance, energy density, retention (power density excluded from ranking)


In [ ]:
def weighted_sum_objective(process_vals, weights, obj_bounds):
    C, E, R = predict_objectives(process_vals)
    # min-max normalize each objective to [0,1] using observed dataset ranges for comparability
    Cn = (C - obj_bounds["C"][0]) / (obj_bounds["C"][1] - obj_bounds["C"][0] + 1e-9)
    En = (E - obj_bounds["E"][0]) / (obj_bounds["E"][1] - obj_bounds["E"][0] + 1e-9)
    Rn = (R - obj_bounds["R"][0]) / (obj_bounds["R"][1] - obj_bounds["R"][0] + 1e-9)
    score = weights[0] * Cn + weights[1] * En + weights[2] * Rn
    return -score  # differential_evolution minimizes

obj_bounds = {
    "C": (df["specific_capacitance_F_g"].min(), df["specific_capacitance_F_g"].max()),
    "E": (df["energy_density_Wh_kg"].min(), df["energy_density_Wh_kg"].max()),
    "R": (df["capacity_retention_pct"].min(), df["capacity_retention_pct"].max()),
}

n_weight_samples = 25   # scan the objective-weight simplex
rng_w = np.random.default_rng(RANDOM_STATE)
raw_weights = rng_w.dirichlet(alpha=np.ones(3), size=n_weight_samples)

pareto_candidates = []
for weights in raw_weights:
    res = differential_evolution(
        weighted_sum_objective, bounds, args=(weights, obj_bounds),
        maxiter=60, popsize=15, seed=RANDOM_STATE, tol=1e-6, polish=True, disp=False,
    )
    C, E, R = predict_objectives(res.x)
    pareto_candidates.append({
        **{f: v for f, v in zip(PROCESS_FEATURES, res.x)},
        "specific_capacitance_F_g": C, "energy_density_Wh_kg": E, "capacity_retention_pct": R,
        "weights": tuple(weights.round(3)),
    })

candidates_df = pd.DataFrame(pareto_candidates)
print(f"Generated {len(candidates_df)} candidate solutions across the weight simplex.")
candidates_df[["specific_capacitance_F_g", "energy_density_Wh_kg", "capacity_retention_pct"]].round(2)


In [ ]:
def is_dominated(row, others, obj_cols, sense="max"):
    """Returns True if `row` is dominated by any row in `others` on all obj_cols."""
    for _, other in others.iterrows():
        better_or_equal = all(other[c] >= row[c] for c in obj_cols)
        strictly_better = any(other[c] > row[c] for c in obj_cols)
        if better_or_equal and strictly_better:
            return True
    return False

obj_cols = ["specific_capacitance_F_g", "energy_density_Wh_kg", "capacity_retention_pct"]
dominated_flags = [
    is_dominated(row, candidates_df.drop(idx), obj_cols)
    for idx, row in candidates_df.iterrows()
]
pareto_front = candidates_df.loc[~np.array(dominated_flags)].sort_values(
    "specific_capacitance_F_g", ascending=False
).reset_index(drop=True)

print(f"Pareto-optimal (non-dominated) solutions found: {len(pareto_front)} / {len(candidates_df)}")
pareto_front[PROCESS_FEATURES + obj_cols].round(3)


In [ ]:
fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.scatter(candidates_df["specific_capacitance_F_g"], candidates_df["energy_density_Wh_kg"],
            candidates_df["capacity_retention_pct"], alpha=0.3, label="dominated", c="gray")
ax1.scatter(pareto_front["specific_capacitance_F_g"], pareto_front["energy_density_Wh_kg"],
            pareto_front["capacity_retention_pct"], alpha=0.9, label="Pareto-optimal", c="crimson", s=50)
ax1.set_xlabel("Capacitance (F/g)"); ax1.set_ylabel("Energy density (Wh/kg)"); ax1.set_zlabel("Retention (%)")
ax1.set_title("3-objective Pareto front")
ax1.legend()

ax2 = fig.add_subplot(1, 2, 2)
ax2.scatter(candidates_df["specific_capacitance_F_g"], candidates_df["energy_density_Wh_kg"],
            c="gray", alpha=0.3, label="dominated")
ax2.scatter(pareto_front["specific_capacitance_F_g"], pareto_front["energy_density_Wh_kg"],
            c="crimson", label="Pareto-optimal")
ax2.set_xlabel("Capacitance (F/g)"); ax2.set_ylabel("Energy density (Wh/kg)")
ax2.set_title("Capacitance vs energy density")
ax2.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pareto_front.png", bbox_inches="tight")
plt.show()


In [ ]:
# Single "best compromise" pick: closest to the ideal point in normalized objective space
ideal = pareto_front[obj_cols].max()
nadir = pareto_front[obj_cols].min()
norm = (pareto_front[obj_cols] - nadir) / (ideal - nadir + 1e-9)
dist_to_ideal = np.sqrt(((1 - norm) ** 2).sum(axis=1))
best_compromise = pareto_front.loc[dist_to_ideal.idxmin()]

print("=== AI-recommended optimal synthesis/fabrication conditions (Step 7 output) ===")
for f in PROCESS_FEATURES:
    print(f"  {f:24s}: {best_compromise[f]:.3f}")
print("--- predicted performance ---")
for c in obj_cols:
    print(f"  {c:28s}: {best_compromise[c]:.2f}")

best_compromise.to_frame("value").to_csv(OUTPUT_DIR / "optimal_conditions.csv")
pareto_front.to_csv(OUTPUT_DIR / "pareto_front.csv", index=False)


## Section 9 — Experimental Validation (Step 8)

Take the AI-recommended `best_compromise` conditions from Section 8, physically synthesize
and test that electrode (Steps 1–6), then enter the **real measured values** below. The cell
compares them against the model's prediction and reports the validation error — this is what
closes the design loop and either confirms the model or flags where it needs refinement.

> Replace the `None` placeholders with your real measured results.


In [ ]:
measured_results = {
    "specific_capacitance_F_g": None,   # <-- enter real GCD-measured value
    "energy_density_Wh_kg": None,       # <-- enter real derived value
    "capacity_retention_pct": None,     # <-- enter real post-cycling value
}

validation_rows = []
for k, measured in measured_results.items():
    predicted = best_compromise[k]
    if measured is None:
        validation_rows.append({"metric": k, "predicted": predicted, "measured": np.nan,
                                 "abs_error": np.nan, "pct_error": np.nan})
    else:
        abs_err = abs(measured - predicted)
        pct_err = 100 * abs_err / max(abs(predicted), 1e-9)
        validation_rows.append({"metric": k, "predicted": predicted, "measured": measured,
                                 "abs_error": abs_err, "pct_error": pct_err})

validation_df = pd.DataFrame(validation_rows)
validation_df.round(3)


In [ ]:
if validation_df["measured"].notna().all():
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(validation_df))
    w = 0.35
    ax.bar(x - w/2, validation_df["predicted"], width=w, label="AI-predicted", color="steelblue")
    ax.bar(x + w/2, validation_df["measured"], width=w, label="experimentally measured", color="crimson")
    ax.set_xticks(x); ax.set_xticklabels(validation_df["metric"], rotation=20, ha="right")
    ax.legend(); ax.set_title("Step 8 — predicted vs. experimentally validated performance")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "validation_comparison.png", bbox_inches="tight")
    plt.show()

    mean_pct_error = validation_df["pct_error"].mean()
    print(f"Mean absolute percentage error (predicted vs measured): {mean_pct_error:.2f}%")
    if mean_pct_error < 10:
        print("-> Model validated: predictions within 10% of experiment.")
    else:
        print("-> Model deviates >10% from experiment: consider retraining with this new "
              "data point appended to the dataset (active-learning loop).")
else:
    print("Enter real measured_results values above to run the quantitative validation.")


## Section 10 — Active-learning refinement loop (optional but recommended)

Once real validation data exists, append it to `df`, regenerate `df_fe`, and retrain
(Sections 3–8) with the enlarged dataset. Repeating this after every few validated
electrodes lets the model's Pareto-front recommendations improve iteratively — a standard
closed-loop materials-discovery pattern.

```python
new_row = {**{f: best_compromise[f] for f in PROCESS_FEATURES}, **measured_results}
# fill in bet_surface_area_m2g, crystallite_size_nm, conductivity_S_cm, power_density_W_kg
# from your real characterization (Step 6) before appending:
# df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
```

## Summary

| Step | Notebook section | Status |
|---|---|---|
| 1–6 (synthesis/characterization) | Section 1 (schema) | replace synthetic data with real CSV |
| 7 — Physics-informed AI modelling | Sections 3–7 | fully implemented (RF, GBR, PINN) |
| 7 — Multi-objective optimization | Section 8 | fully implemented (weighted-sum + Pareto filtering) |
| 8 — Experimental validation | Section 9 | template — enter real measurements |
